```
<03_aihub.ipynb>

제미나이 의존도: 80-90%

처음으로 AI HUB 사이트에서 데이터를 가져와서 모델 학습을 해보았다.
캐글처럼 폴더 구조가 깔끔하지 않아서 내가 따로 손을 봐줘야 하는 상황이었다.
사과 약 3600개가 있었고, 라벨링 데이터(txt)는 아니고 json 파일이 있었다.
그래서 라벨링 데이터를 만드는데에 필요한 width, height, xmin 등의 정보만 가져와서 txt 파일로 만들었다.
일단 맛보기로 50개의 데이터만 가져와서 테스트하였고, 학습은 성공적이었다.
다음엔 train / val / test로 잘누어서 해보고 싶다.
```

In [1]:
!pwd

/Users/jeongjaehun/Github/03_object_detection


In [2]:
import os
common_path = "images/apple_from_aihub/01.데이터/1.Training"
json_path = os.path.join(common_path, "라벨링데이터_230921_add/Apple_fuji_L")
image_path = os.path.join(common_path, "원천데이터_230921_add/Apple_fuji_L")
print(json_path)
print(image_path)

images/apple_from_aihub/01.데이터/1.Training/라벨링데이터_230921_add/Apple_fuji_L
images/apple_from_aihub/01.데이터/1.Training/원천데이터_230921_add/Apple_fuji_L


In [19]:
import json
test_basename = "apple_fuji_L_1-1_1TOP"
with open(os.path.join(json_path, test_basename + ".json")) as f:
    data = json.load(f)

# print(data.keys())
# print(data.values())

# print(json.dumps(data, indent=4, ensure_ascii=False))

print(f"img_height: {data['img_height']}")
print(f"img_width: {data['img_width']}")

bndbox = data['bndbox']
xmin = bndbox['xmin']
ymin = bndbox['ymin']
xmax = bndbox['xmax']
ymax = bndbox['ymax']
print(f"xmin: {xmin}, ymin: {ymin}, xmax: {xmax}, ymax: {ymax}")


img_height: 1000
img_width: 1000
xmin: 0, ymin: 0, xmax: 1000, ymax: 1000


음.. 일단 정리되지 않은 말을 써보자면.
shutil로 정렬된 이미지 중에서 앞 50개를 가져온다.
그러면서, 끝 확장자만 json으로 바꾸어서, json 파일도 있다면 이미지와 같이 가져온다.(없으면 둘 다 안가져옴.)

json 폴더를 새로 만들어서.. 거기에 집어넣는다.

여기까지 오면 이미지 50개, json 50개가 나올 것이다.
json 50개 내부를 뜯어서 모델 학습에 필요한 txt 파일로 새로 만들어서 labels 폴더에 넣는다.


In [28]:
# YOLOv8 학습용 디렉토리 생성
dir_path = "images/apples/train/jsons/"

os.makedirs(dir_path, exist_ok=True)

In [75]:
# json 파일들을 정렬해서 앞 50개만 가져온다.
import shutil
from pathlib import Path
images = os.listdir(image_path)
jsons = os.listdir(json_path)
# print(len(images))
# print(len(jsons))
images.sort()
jsons.sort()

dest_image_path = "images/apples/train/images"
dest_json_path = "images/apples/train/jsons"
dest_label_path = "images/apples/train/labels"

for i, image_name in enumerate(images[:50]):
    # print(image_name)
    json_name = image_name.replace('.png', '.json')
    
    full_json_path = os.path.join(json_path, json_name)
    full_image_path = os.path.join(image_path, image_name)
    
    if os.path.isfile(full_json_path):
        # print(f"존재")
        # print(full_json_path)
        # print(os.path.join(dest_json_path, json_name))
        dest_json_file_path = os.path.join(dest_json_path, json_name)
        shutil.copy(full_json_path, dest_json_file_path)

        dest_image_file_path = os.path.join(dest_image_path, image_name)
        # print(full_image_path)
        # print(dest_image_file_path)
        shutil.copy(full_image_path, dest_image_file_path)

        with open(dest_json_file_path) as f:
            pure_name = Path(dest_json_file_path).stem
            # print(pure_name)
            data = json.load(f)
            img_height = data['img_height']
            img_width = data['img_width']
            xmin = data['bndbox']['xmin']
            ymin = data['bndbox']['ymin']
            xmax = data['bndbox']['xmax']
            ymax = data['bndbox']['ymax']

            w = xmax - xmin
            h = ymax - ymin
            cx = xmin + (w / 2)
            cy = ymin + (h / 2)

            yolo_cx = cx / img_width
            yolo_cy = cy / img_height
            yolo_w = w / img_width
            yolo_h = h / img_height
            # print(yolo_cx, yolo_cy, yolo_w, yolo_h)

            save_path = os.path.join(dest_label_path, pure_name + ".txt")
            with open(save_path, "w", encoding="utf-8") as f:
                f.write(f"0 {yolo_cx} {yolo_cy} {yolo_w} {yolo_h}")

print("Complete")

Complete


In [74]:
with open("images/apples/data.yaml", "w", encoding="utf-8") as f:
    f.write(f"train: ../train/images\n")
    f.write(f"val: ../train/images\n")
    f.write("\n")
    f.write(f"nc: 1\n")
    f.write(f"names: ['apple']")

In [71]:
def cleaner(dataset_root):
    # dataset_root = "images/apples"
    print("데이터셋 내부의 .ipynb_checkpoints 청소 시작.")
    
    for root, dirs, files in os.walk(dataset_root):
        if ".ipynb_checkpoints" in dirs:
            target_dir = os.path.join(root, ".ipynb_checkpoints")
            shutil.rmtree(target_dir)
            print(f"제거 완료: {target_dir}")

In [76]:
cleaner("images/apples")

데이터셋 내부의 .ipynb_checkpoints 청소 시작.
제거 완료: images/apples/.ipynb_checkpoints


In [78]:
from ultralytics import YOLO

current_dir = os.getcwd()

model = YOLO('yolov8n.pt')

results = model.train(
    data="images/apples/data.yaml", 
    epochs=5, 
    imgsz=640, 
    device='mps', 

    project=os.path.join(current_dir, "runs"), 
    name="apples"
)



Ultralytics 8.4.87 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=images/apples/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=apples, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo